# YouTube Crawler Testing Notebook

This notebook provides interactive testing of the Travel AI YouTube crawler.

## Contents
1. Setup and Configuration
2. Test Single Video Crawl
3. Explore Video Data Structure
4. Visualize Transcript Timeline
5. Test Multiple Videos
6. Test S3 Upload (Optional)
7. Data Analysis

## 1. Setup and Configuration

In [ ]:
# Import required libraries
import sys
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import crawler modules
from src.crawlers.youtube import crawl_video, crawl_videos, extract_video_id
from src.utils.config import config
from src.utils.logging import setup_logging, get_logger
from src.utils.schemas import YouTubeVideo
from src.storage.s3 import S3Storage

print("✓ All imports successful")

In [ ]:
# Setup logging
setup_logging("INFO")
logger = get_logger(__name__)

print("✓ Logging configured")

In [ ]:
# Check configuration
if config:
    print("Configuration loaded:")
    print(f"  Log Level: {config.LOG_LEVEL}")
    print(f"  Rate Limit: {config.CRAWLER_RATE_LIMIT}s")
    print(f"  Max Retries: {config.MAX_RETRIES}")
    print(f"  AWS Region: {config.AWS_REGION}")
    print(f"  S3 Bucket: {config.S3_BUCKET_NAME}")
else:
    print("⚠ Configuration not loaded - check .env file")

## 2. Test Single Video Crawl

Let's crawl a single YouTube video and examine the results.

In [ ]:
# Test URL - Boracay travel vlog
test_url = "https://www.youtube.com/watch?v=Jx3_JJVhnPo"

print(f"Crawling: {test_url}")
print("This may take 10-20 seconds...\n")

video = crawl_video(test_url)

if video:
    print("✓ Video crawled successfully!\n")
    print(f"Title: {video.title}")
    print(f"Author: {video.author}")
    print(f"Duration: {video.duration_seconds}s ({video.duration_seconds // 60}m {video.duration_seconds % 60}s)")
    print(f"Published: {video.published_date}")
    print(f"Views: {video.metadata.view_count:,}")
    print(f"Language: {video.language}")
    print(f"Transcript segments: {len(video.transcript)}")
    print(f"Tags: {', '.join(video.metadata.tags[:5])}")
else:
    print("✗ Failed to crawl video (may not have English transcript)")

## 3. Explore Video Data Structure

Let's examine the complete data structure of the crawled video.

In [ ]:
if video:
    # Convert to dictionary
    video_dict = video.to_dict()
    
    print("Video Data Structure:")
    print("=" * 50)
    
    # Show top-level fields
    for key, value in video_dict.items():
        if key not in ['transcript', 'metadata']:  # Skip large nested fields
            print(f"{key}: {value}")
    
    print("\nMetadata:")
    for key, value in video_dict['metadata'].items():
        if key != 'tags':
            print(f"  {key}: {value}")
    print(f"  tags: {video_dict['metadata']['tags'][:3]}...")
    
    print(f"\nTranscript: {len(video_dict['transcript'])} segments")
    print("First 3 segments:")
    for i, seg in enumerate(video_dict['transcript'][:3]):
        print(f"  [{i}] {seg['start']:.1f}s: {seg['text'][:50]}...")

In [ ]:
# Get full transcript text
if video:
    full_text = video.get_full_transcript_text()
    
    print("Full Transcript Text:")
    print("=" * 50)
    print(f"Total characters: {len(full_text):,}")
    print(f"Total words: {len(full_text.split()):,}")
    print("\nFirst 500 characters:")
    print(full_text[:500] + "...")

## 4. Visualize Transcript Timeline

Create a visualization of when different parts of the transcript occur.

In [ ]:
if video:
    # Create DataFrame from transcript
    transcript_data = []
    for seg in video.transcript:
        transcript_data.append({
            'start': seg.start,
            'end': seg.start + seg.duration,
            'duration': seg.duration,
            'text': seg.text,
            'word_count': len(seg.text.split())
        })
    
    df = pd.DataFrame(transcript_data)
    
    print("Transcript Statistics:")
    print("=" * 50)
    print(f"Total segments: {len(df)}")
    print(f"Total duration: {df['end'].max():.1f}s ({df['end'].max() / 60:.1f}m)")
    print(f"Average segment duration: {df['duration'].mean():.2f}s")
    print(f"Average words per segment: {df['word_count'].mean():.1f}")
    print(f"Total words: {df['word_count'].sum():,}")
    
    # Display sample
    print("\nSample segments:")
    display(df.head(10))

In [ ]:
if video:
    # Plot transcript timeline
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    # Plot 1: Segment duration over time
    axes[0].bar(df['start'], df['duration'], width=df['duration'], alpha=0.7, color='steelblue')
    axes[0].set_xlabel('Time (seconds)')
    axes[0].set_ylabel('Segment Duration (seconds)')
    axes[0].set_title('Transcript Segment Duration Over Time')
    axes[0].grid(axis='y', alpha=0.3)
    
    # Plot 2: Words per segment over time
    axes[1].plot(df['start'], df['word_count'], marker='o', markersize=2, alpha=0.6, color='coral')
    axes[1].set_xlabel('Time (seconds)')
    axes[1].set_ylabel('Words per Segment')
    axes[1].set_title('Words per Segment Over Time')
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n📊 Timeline visualization created")

## 5. Test Multiple Videos

Crawl multiple videos from the sample URLs file.

In [ ]:
# Load sample URLs
sample_urls_file = project_root / "data" / "sample_urls.txt"

if sample_urls_file.exists():
    with open(sample_urls_file, 'r') as f:
        urls = [line.strip() for line in f if line.strip() and not line.startswith('#')]
    
    print(f"Loaded {len(urls)} URLs from sample_urls.txt")
    for i, url in enumerate(urls, 1):
        print(f"  {i}. {url}")
else:
    print("⚠ Sample URLs file not found")
    urls = []

In [ ]:
# Crawl first 3 videos (to avoid taking too long)
if urls:
    print("Crawling first 3 videos...")
    print("This will take about 30-60 seconds...\n")
    
    successful, failed = crawl_videos(
        urls=urls[:3],
        rate_limit=2.0,
        max_retries=2,
        language="en"
    )
    
    print(f"\n✓ Crawl complete!")
    print(f"  Successful: {len(successful)}")
    print(f"  Failed: {len(failed)}")
    
    if successful:
        print("\nSuccessfully crawled videos:")
        for v in successful:
            print(f"  - {v.title} ({len(v.transcript)} segments)")
    
    if failed:
        print("\nFailed URLs:")
        for url in failed:
            print(f"  - {url}")

## 6. Test S3 Upload (Optional)

**Note:** Uncomment and run only if you have AWS credentials configured.

In [ ]:
# # Test S3 upload (uncomment to run)
# if video and config:
#     try:
#         print("Testing S3 upload...")
#         storage = S3Storage()
#         
#         # Upload single video
#         video_dicts = [video.to_dict()]
#         s3_uri = storage.upload_jsonl(
#             data=video_dicts,
#             source="youtube",
#             data_type="test_videos"
#         )
#         
#         print(f"✓ Uploaded to: {s3_uri}")
#         
#         # Test download
#         print("Testing download...")
#         downloaded = storage.download_jsonl(s3_uri)
#         print(f"✓ Downloaded {len(downloaded)} videos")
#         
#     except Exception as e:
#         print(f"✗ S3 test failed: {e}")
# else:
#     print("Skipping S3 test (no video or config)")

print("S3 upload test commented out by default")
print("Uncomment the code above to test S3 functionality")

## 7. Data Analysis

Analyze the crawled videos.

In [ ]:
if 'successful' in locals() and successful:
    # Create summary DataFrame
    summary_data = []
    for v in successful:
        summary_data.append({
            'title': v.title,
            'author': v.author,
            'duration_seconds': v.duration_seconds,
            'duration_minutes': round(v.duration_seconds / 60, 1),
            'views': v.metadata.view_count,
            'transcript_segments': len(v.transcript),
            'transcript_duration': round(v.get_transcript_duration(), 1),
            'words': len(v.get_full_transcript_text().split()),
            'tags': len(v.metadata.tags)
        })
    
    summary_df = pd.DataFrame(summary_data)
    
    print("Video Summary:")
    print("=" * 80)
    display(summary_df)
    
    print("\nStatistics:")
    print(f"  Total videos: {len(summary_df)}")
    print(f"  Total duration: {summary_df['duration_minutes'].sum():.1f} minutes")
    print(f"  Total views: {summary_df['views'].sum():,}")
    print(f"  Total words: {summary_df['words'].sum():,}")
    print(f"  Average duration: {summary_df['duration_minutes'].mean():.1f} minutes")
    print(f"  Average segments: {summary_df['transcript_segments'].mean():.0f}")
else:
    print("No videos to analyze. Run the crawl cells above first.")

In [ ]:
if 'successful' in locals() and successful and len(successful) > 1:
    # Create comparison visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Duration comparison
    axes[0].barh(summary_df['title'].str[:30], summary_df['duration_minutes'], color='steelblue')
    axes[0].set_xlabel('Duration (minutes)')
    axes[0].set_title('Video Duration Comparison')
    axes[0].grid(axis='x', alpha=0.3)
    
    # Plot 2: Transcript segments comparison
    axes[1].barh(summary_df['title'].str[:30], summary_df['transcript_segments'], color='coral')
    axes[1].set_xlabel('Number of Segments')
    axes[1].set_title('Transcript Segments Comparison')
    axes[1].grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n📊 Comparison visualization created")

## 8. Export Results

Save crawled videos to local JSONL file.

In [ ]:
if 'successful' in locals() and successful:
    # Create output directory
    output_dir = project_root / "data" / "raw"
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Generate filename
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = output_dir / f"notebook_test_{timestamp}.jsonl"
    
    # Write JSONL
    with open(output_file, 'w', encoding='utf-8') as f:
        for v in successful:
            json_str = json.dumps(v.to_dict(), ensure_ascii=False)
            f.write(json_str + '\n')
    
    print(f"✓ Saved {len(successful)} videos to: {output_file}")
    print(f"  File size: {output_file.stat().st_size / 1024:.1f} KB")
else:
    print("No videos to export")

## Summary

This notebook demonstrated:
1. ✅ Setting up the crawler environment
2. ✅ Crawling single and multiple videos
3. ✅ Exploring video data structure
4. ✅ Visualizing transcript timelines
5. ✅ Analyzing crawled data
6. ✅ Exporting results to JSONL

Next steps:
- Use `cli/crawl.py` for production crawling
- Upload results to S3 for storage
- Process transcripts with LLMs (Stage 2)